# 002: Understanding the Framework Layer (V1 Pattern G)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from earlysign.core.ledger import Ledger
import ibis

connection = ibis.connect("duckdb://:memory:")
ledger = Ledger(connection, "example_v1").bind(exp_id="exp_v1_001")
ledger.ensure()
ledger

## Tier 0: Ingesting Raw Evidence

In V1 (Pattern G), we use `Session` to define the scientific horizon and `Ingest` to record raw evidence.

In [ ]:
from earlysign.v1.framework.session import Session
from earlysign.v1.methods.actions import Ingest
from pydantic import BaseModel


class Observation(BaseModel):
    value: float
    variant: str


with Session(ledger) as sess:
    Ingest(sess, Observation(value=10.0, variant="C"))
    Ingest(sess, Observation(value=12.0, variant="T"))

ledger.show()

## Tier 1: State Reconstruction (Projectors)

To read data, we use `Projectors` within a `Session`. Projectors translate raw events into structured scientific state while tracking provenance.

In [ ]:
from earlysign.v1.framework.projector import Projector, ProjectionResult
from earlysign.v1.framework.trace import TraceHash


class MeanProjector(Projector[dict]):
    def project(self, table: ibis.Expr) -> ProjectionResult[dict]:
        matched = table.filter(table.payload_type == "Observation")
        pdf = matched.execute()
        if pdf.empty:
            return ProjectionResult(data={"mean": 0.0}, trace=[])

        mean_val = pdf["payload"].apply(lambda x: x["value"]).mean()
        # Extract trace hashes from labels
        traces = [
            TraceHash(str(h))
            for h in pdf["labels"].apply(lambda x: x.get("trace_hash"))
            if h
        ]
        return ProjectionResult(data={"mean": mean_val}, trace=traces)


with Session(ledger) as sess:
    traced_mean = sess.Read(MeanProjector())
    print(f"Hydrated State: {traced_mean.data}")
    print(f"Implicit Trace (accumulated by Read): {sess.trace}")

## Tier 2: Analytical Operations (CommitCallResult)

We can perform analytical operations and commit their results. The lineage is automatically tracked from the inputs.

In [ ]:
from earlysign.v1.framework.write_models import WriteModel


class AnalysisResult(BaseModel):
    doubled_mean: float


def analyze_mean(summary: dict):
    return {"doubled_mean": summary["mean"] * 2}


with Session(ledger) as sess:
    traced_mean = sess.Read(MeanProjector())

    result = WriteModel.CommitCallResult(
        sess, AnalysisResult, analyze_mean, summary=traced_mean
    )
    print(f"Committed Result: {result}")

ledger.show()

# Multi-Step Orchestration: Ledger Concept via Template API

We now demonstrate the procedure of a group sequential test using the high-level Template API.
This includes performing updates and explicitly calling specialized Progress and Final reports.

In [ ]:
import numpy as np
from earlysign.v1.methods.group_sequential.protocol import GSTProtocol
from earlysign.v1.methods.group_sequential.design_planner import DesignPlanner
from earlysign.v1.templates.binomial_ab import BinomialABTemplate
from earlysign.v1.methods.binomial import BatchObservation

# Clear and bind a new experiment
ledger = Ledger(ibis.connect("duckdb://:memory:"), "events").bind(
    experiment="reporting_v1"
)
ledger.ensure()

## Step 1: Initialize Protocol and Design

We define the design and initialize the template using a unified `set_protocol` API.

In [ ]:
protocol = GSTProtocol(
    alpha=0.05, K=5, spending_function="rho_family", rho=3.0, side=1, delta=0.03
)
trial = BinomialABTemplate(ledger)

# 1. Record the initial scientific intent (Planning Phase)
trial.set_protocol(protocol)

planner = DesignPlanner()
plan = planner.plan_binomial_ab(
    alpha=protocol.alpha,
    power=protocol.power,
    p_control=0.10,
    delta=protocol.delta,
    k=protocol.K,
)

# 2. Realize the design and record the structural update (Execution Phase)
protocol.n_max = plan["n_max"]
protocol.milestones = [0.2, 0.4, 0.6, 0.8, 1.0]
protocol.boundaries = plan["boundaries"]
trial.set_protocol(protocol)

print(f"Planned N_max: {plan['n_max']}")

## Step 2: Multi-Look Simulation with On-demand Reporting

We simulate the study. We call `update()` for each batch and explicitly call report methods when a stop/look is reached.

In [ ]:
p_c, p_t = 0.10, 0.15
n_per_batch = plan["n_max"] // 5

for look in range(1, 6):
    print(f"--- Processing Look {look} ---")
    s_c = np.random.binomial(n_per_batch, p_c)
    s_t = np.random.binomial(n_per_batch, p_t)

    batch = [
        BatchObservation(n=n_per_batch, success=s_c, variant="C"),
        BatchObservation(n=n_per_batch, success=s_t, variant="T"),
    ]

    # update() performs analysis and returns minimal status info
    res = trial.update(batch)
    print(f"Update result: {res}")

    if res["look"]:
        # Explicitly call progress_report() for detailed interim info
        report = trial.progress_report()
        print(f"Detailed Progress Report: {report}")

        if res["status"] == "STOP_EFFICACY":
            print(">>> STOPPED EARLY!")
            # Explicitly call final_report() at the end
            final = trial.final_report(is_rejected=True, final_status="STOP_EFFICACY")
            print(f"Detailed Final Report: {final}")
            break
        elif look == 5:
            final = trial.final_report(is_rejected=False, final_status="COMPLETED")
            print(f"Detailed Final Report: {final}")